# Versuch 1 - Auswertung

Dieses Notebook dient dem systematischen herausschreiben der relevanten Codestellen des originalen Programms von Chiara Weckmann ```MSR_map_coil.ipynb```.
___
Erstellt am 29.Apr.2026 von Gregor Bock

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import scipy
from scipy.interpolate import griddata
import bfieldtools
from bfieldtools.utils import load_example_mesh,combine_meshes
import trimesh
import os
mu_0=scipy.constants.mu_0
pi=scipy.constants.pi

# Self made functions and classes
import Ausgelagerte_Funktionen_Versuchsauswertung as fkt
from Externe_Classes import Coil_Layup, Mu_material

## Load Experiment Data

This data was obtained on 27.Apr.2026. Therefore, coils were used on the left and right side of the MSR. They had a diameter of $d=0.34$ m, a horizontal distance of $d_x = 0.507$ m and a verticall distance of $d_z = 0.553$ m. The distances were measured with respect to the upper corners of the inner (wooden) room-side with the door.

In [ ]:
# Specify folder path for .npz and points folder (.npz file should contain L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)
folder_path_points_no_current = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\after_degauss_no_current_2026-04-27_16-07-03\map\points"
folder_path_points_with_current = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\after_degauss_with_current_2026-04-27_16-54-25\map\points"

# If .npz file does not hold geometric track data, specify it here
L_x = 0.800                                         # length of the mapped volume in x-direction
L_y = 0.800                                         # length of the mapped volume in y-direction
L_z = 0.400                                         # length of the mapped volume in z-direction
step_size = 0.200                                   # size of the grid steps

shift_x = 0                                         # shift of the mapped volume in x-direction
shift_y = 0                                         # shift of the mapped volume in y-direction
shift_z = 0                                         # shift of the mapped volume in z-direction

# Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
target_point_coord, B_target_point_no_current, B_target_point_with_current = fkt.load_data_from_folder(folder_path_points_no_current, folder_path_points_with_current, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)

## Plot the extracted Data

Plot each $B$-field component as well as the norm of the $B$-field of both maps (with and without coil current) directly next to each other for better comparison.

In [ ]:
x = target_point_coord[:, 0]
y = target_point_coord[:, 1]
z = target_point_coord[:, 2]

Bx_no_I = B_target_point_no_current[:, 0]
By_no_I = B_target_point_no_current[:, 1]
Bz_no_I = B_target_point_no_current[:, 2]
Bnorm_no_I = np.linalg.norm(B_target_point_no_current, axis=1)

Bx_with_I = B_target_point_with_current[:, 0]
By_with_I = B_target_point_with_current[:, 1]
Bz_with_I = B_target_point_with_current[:, 2]
Bnorm_with_I = np.linalg.norm(B_target_point_with_current, axis=1)

fig = plt.figure(figsize=(18, 28))
axs = [
    fig.add_subplot(4, 2, 1, projection='3d'),
    fig.add_subplot(4, 2, 3, projection='3d'),
    fig.add_subplot(4, 2, 5, projection='3d'),
    fig.add_subplot(4, 2, 7, projection='3d'),
    fig.add_subplot(4, 2, 2, projection='3d'),
    fig.add_subplot(4, 2, 4, projection='3d'),
    fig.add_subplot(4, 2, 6, projection='3d'),
    fig.add_subplot(4, 2, 8, projection='3d')
]

component_data = [Bx_no_I, By_no_I, Bz_no_I, Bnorm_no_I, Bx_with_I, By_with_I, Bz_with_I, Bnorm_with_I]
v_min = min(d.min() for d in component_data)
v_max = max(d.max() for d in component_data)
titles = [
    r'$B_x$ without current [T]',
    r'$B_y$ without current [T]',
    r'$B_z$ without current [T]',
    r'$|B|$ without current [T]',
    r'$B_x$ with current [T]',
    r'$B_y$ with current [T]',
    r'$B_z$ with current [T]',
    r'$|B|$ with current [T]'
]
plots = [
    (Bx_no_I, r'$B_x$ [T]', v_min, v_max),
    (By_no_I, r'$B_y$ [T]', v_min, v_max),
    (Bz_no_I, r'$B_z$ [T]', v_min, v_max),
    (Bnorm_no_I, r'$|B|$ [T]', v_min, v_max),
    (Bx_with_I, r'$B_x$ [T]', v_min, v_max),
    (By_with_I, r'$B_y$ [T]', v_min, v_max),
    (Bz_with_I, r'$B_z$ [T]', v_min, v_max),
    (Bnorm_with_I,r'$|B|$ [T]', v_min, v_max)
]
for ax, (data, label, vmin, vmax), title in zip(axs, plots, titles):
    sc = ax.scatter(x, y, z, c=data, s=100, cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel(r'$x$ [m]')
    ax.set_ylabel(r'$y$ [m]')
    ax.set_zlabel(r'$z$ [m]')
    plt.colorbar(sc, ax=ax, label=label)

# fig.suptitle(f'First Measurement of coil influence on B-field at target points', fontsize = 20)
plt.tight_layout()
plt.show()

## Geometric Data of MSR

All geometric data of the MSR as well the coil Layup is given in meters and ampere (generally only SI-base units)!

In [ ]:
# Inner shield
shield_dim = 2.3            # length, width and height of the inner most shield
shield_thickness = 1e-3     # thickness of the shield walls

# Door
door_width = 0.95           # (width of the door)
door_height = 2.004         # (height of the door)
door_offset_x = 0.2375      # (distance from door frame to right wall)
door_floor_offset = 0.0     # (distance from door frame to floor)

# Inner wood structure
height_inner_wood = 2.20    # height of the inner wooden structure
width_inner_wood = 2.34     # width of the inner wooden structure
depth_inner_wood = 2.34     # depth of the inner wooden structure

# Coil Layout
coil_diameter = 0.34                # diameter of the coils
z_dist = 0.506                      # vertical distance between coils
x_dist = 0.553                      # horizontal distance between coils (x-direction)
y_dist = 0.553                      # horizontal distance between coils (y-direction)
coil_plane_dist_to_origin_x = 1.5   # distance from the center of the room to the plane where the coils are placed in x-direction
coil_plane_dist_to_origin_y = 1.5   # distance from the center of the room to the plane where the coils are placed in y-direction
coil_plane_dist_to_origin_z = 1.5   # distance from the center of the room to the plane where the coils are placed in z-direction
usable_length_x = 2.34              # usable length in x-direction for coil placement
usable_length_y = 2.34              # usable length in y-direction for coil placement
usable_length_z = 2.20              # usable length in z-direction for coil placement
n_windings = 10                     # number of windings per coil
current = 1.0                       # current in A flowing through the coils
# Note: One can give a scalar input for n_windings, which will be applied to every coil
#       or a list of integer numbers, which need to has the same length as coils present in the layout (if the shapes do not match, an error accours!)


## Call instance to Coil_Layup class

The class ```Coil_Layup``` from the file **Externe_Classes.py** is called to give the coil geometries by inputting all the geometric data.

Furthermore, this block plots the coils which were specified by the geomertic data as well as the centers of these coils.

In [ ]:
Coils = Coil_Layup(coil_diameter, x_dist, y_dist, z_dist, n_windings, current, coil_plane_dist_to_origin_x, coil_plane_dist_to_origin_y, coil_plane_dist_to_origin_z, usable_length_x, usable_length_y, usable_length_z)
all_circles = Coils.all_coils  # Shape: (n_coils_xy, 100, 3)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for k in range(all_circles.shape[0]):
    ax.plot(all_circles[k, :, 0], all_circles[k, :, 1], all_circles[k, :, 2], color='blue')
plt.show()

fig_grid = plt.figure()
ax_grid = fig_grid.add_subplot(111, projection='3d')
ax_grid.scatter(Coils.grid[:, 0], Coils.grid[:, 1], Coils.grid[:, 2], color='red', s=100)

print(all_circles.shape)


## Predicted $B$-field - only coils, no $\mu$-metal

By calling the method ```coil_field_by_Biot_Savart``` one can calculate the $B$-field which is produced by the coil layup without the influence of the $\mu$-metal and without consideration of the residual field.

This block also plots the result.

In [ ]:
B_coil_predicted = Coils.coil_field_by_Biot_Savart(target_point_coord)

Bmag = np.linalg.norm(B_coil_predicted, axis=1)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(
    target_point_coord[:, 0],
    target_point_coord[:, 1],
    target_point_coord[:, 2],
    c=Bmag,
    s=100,
    cmap='viridis'
)
fig.colorbar(sc, ax=ax, label=r'$|\mathbf{B}|$')

## Shield meshing class

To account for the $\mu$-metal a meshed geometry accourding to the specified geometric data can be created by calling the class ```Mu_material``` from the file **Externe_Classes.py**. This will output a triangular mesh of the $\mu$-metal surface.

This block also plots the mesh (of one planar surface), the mesh boundaries (should be an empty plot) as well as the cells in 3D just as in the original programm.

In [ ]:
mu_grid = Mu_material(shield_dim, shield_thickness)

# # From here on, plotting starts!!!
# # plot triangular grid of one planar surface
# fig, ax = plt.subplots(figsize=(6,6))
# ax.triplot(mu_grid.triang_shield, color='k', linewidth=0.8)

# # plot meshboundaries of the mu-metal
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# inner_idx_shield=bfieldtools.utils.find_mesh_boundaries(mu_grid.total_shield)
# inner_vertex_idx_shield=inner_idx_shield[0]
# for idx in inner_vertex_idx_shield:
#     boundary=mu_grid.total_shield.vertices[idx]
#     ax.scatter(boundary[:,0],boundary[:,1],boundary[:,2])

# # plot vertices
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# verts_shield = mu_grid.total_shield.vertices
# ax.scatter(verts_shield[:,0],verts_shield[:,1],verts_shield[:,2],alpha=0.5)

# # plot vertices of the shield and points inside the shield
# points_inside = mu_grid.total_shield.vertices - mu_grid.shield_thickness * mu_grid.total_shield.vertex_normals
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(verts_shield[:,0],verts_shield[:,1],verts_shield[:,2],alpha=0.3)
# ax.scatter(points_inside[:,0],points_inside[:,1],points_inside[:,2],alpha=0.3)

## Mesh the coil plane as well

In [ ]:
total_planes = Coils.create_mesh(door_width, door_height, door_floor_offset, door_offset_x)

# Plot the mesh in 2D
x = Coils.coil_plus_xz_with_gap.vertices[:, 0]
z = Coils.coil_plus_xz_with_gap.vertices[:, 2]
faces = Coils.coil_plus_xz_with_gap.faces

triang = mtri.Triangulation(x, z, faces)
fig, ax = plt.subplots(figsize=(6, 6))
ax.triplot(triang, color='k', linewidth=0.8)
ax.set_aspect('equal')
ax.set_title('XZ wall with door')
plt.show()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
inner_idx=bfieldtools.utils.find_mesh_boundaries(Coils.total_planes)
inner_vertex_idx=inner_idx[0]
for idx in inner_vertex_idx:
    boundary=Coils.total_planes.vertices[idx]
    ax.scatter(boundary[:,0],boundary[:,1],boundary[:,2])

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
#fig.subplots_adjust(left=0.0, right=30.0, bottom=0.0, top=1.0)
verts=Coils.total_planes.vertices
ax.scatter(verts[:,0],verts[:,1],verts[:,2],alpha=0.1,label='coil planes')
ax.scatter(target_point_coord[:,0],target_point_coord[:,1],target_point_coord[:,2],label='MSR Map Points')
# Axis labels
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_zlabel('z [m]')
plt.legend()
#plt.tight_layout()
plt.show()

## Finding the Coupling-matrix $C_{ij}$

Workflow of the program:
- Calculate scalar potential matrices $U_{coil}$ and $U_{shield}$:
\begin{equation}
    \Phi(r_i) = \sum_{j=i}^{N_{vertices}} U_{ij}\psi_j
\end{equation}
where $\Phi(r_i)$ is the scalar potential, $\psi_j$ is the stream function at vertex $j$ and U_{ij} is the scalar potential matrix.

- From scalar potential matrices $U_{ij} compute coil-shiel-coupling matrix $M_{ij}$
\begin{equation}
    U_{shield} \cdot M_{tot\_shield} = U_{coils}
\end{equation}

- Calculate the magnetic coupling matrix $C_{ij\alpha}$ of the $\mu$-metal in all three space directions
\begin{align}
    B_{\alpha,\text{shield}}(r_k) = \sum_{i=1}^{N_{vertices}} C_{ki\alpha,\text{shield}}\psi_{i,\text{shield}} && \alpha \in \{x,y,z\}
\end{align}
where $B_{\alpha,\text{shield}}(r_k)$ is the magnetic field at $r_k$ in direction $\alpha$ produced by the shield, and $\psi_{i,\text{shield}}$ is the streamfunction in the shield at vertex $i$.

- Calculate the magnetic coupling matrix $C^*_{ij\alpha\text{shield}}$ of the $\mu$-metal as if it was caused by the coil-plane (change of reference)
 \begin{equation}
    C^*_{\alpha,\text{shield}} = C_{\alpha,\text{shield}} \cdot M_{tot\_shield}
 \end{equation}

- Calculate the magnetic coupling matrix $C^*_{ij\alpha\text{coil}}$ of the coils
\begin{align}
    B_{\alpha,\text{coil}}(r_k) = \sum_{i=1}^{N_{vertices}} C_{ki\alpha,\text{coil}}\psi_{i,\text{coil}} && \alpha \in \{x,y,z\}
\end{align}
where all quantities are analog to those of the shield case.

- At the boundaries of the coil plane, it holds that in any case no current can flow over the edge, resulting in a streamfunction which is constant and can be set to zero (gauge). (**Note**, that since the shield of the MSR is connected at the edges, allowing current-flow over the edge. Therefore the edges of the shield are no boundaries! -> <span style="color:red">Is currentflow conserved at the edges? How does the program know, which cells are adjacent around the corner?</span>)

- Optain the total coupling matrix $C_{ij\alpha, \text{total}}$ of both coils and shield by adding both together
\begin{equation}
    C_{ij\alpha, \text{total}} = C^*_{ij\alpha,\text{shield}} + C_{ij\alpha,\text{coil}}
\end{equation}

- Reshape the coil coupling matrix $C^*_{ij\alpha\text{coil}}$ and the total coupling matrix $C_{ij\alpha, \text{total}}$ to better use it later on (shape: N, 3, M -> 3N, M)



In [ ]:
# Scalar-Potential Coupling Matrix U_{ij} (only dependent on geometrical data of meshes)
    # mu-material
U_coupling_shield_inside = bfieldtools.mesh_magnetics.scalar_potential_coupling(mu_grid.total_shield, mu_grid.points_inside)
    # Coils
U_coupling_coils_inside = bfieldtools.mesh_magnetics.scalar_potential_coupling(Coils.total_planes, mu_grid.points_inside) 

# total shield coupling which links the coil plane to the shield plane U_shield * M = U_coil
Coil_Shield_coupling = np.linalg.solve(U_coupling_shield_inside, U_coupling_coils_inside)

# B-field coupling matrix C_shield of the shield alone (only dependent on gemetrical data of mu-mesh and target point coordinates)
Coupling_shield = bfieldtools.mesh_magnetics.magnetic_field_coupling(mu_grid.total_shield, target_point_coord)

# B-field coupling matrix C*_shield of the shield alone, seen as a secondary source from the coil plane perspective C*_shield = C_shield * M
secondary_Coupling_shield = np.tensordot(Coupling_shield, Coil_Shield_coupling, axes=([2],[0]))

# B-field coupling matrix C_coil of the coil plane (only dependent on gemetrical data of coil-mesh and target point coordinates)
Coupling_coil = bfieldtools.mesh_magnetics.magnetic_field_coupling(Coils.total_planes, r=target_point_coord)

# Note: Not sure if this is correct!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! -> see Markdown above!
inner_idx = bfieldtools.utils.find_mesh_boundaries(Coils.total_planes)
inner_vertex_idx = inner_idx[0]
for idx in inner_vertex_idx:
    Coupling_coil[:, :, idx] = 0

# Combined coupling matrix C_total which incorporates the effect of the shield as a secondary source and the coil plane as a primary source
total_Coupling = Coupling_coil + secondary_Coupling_shield

# # Suppose C has shape (N,3,M)
N, _, M = Coupling_coil.shape
Coupling_coil_flat = Coupling_coil.reshape(3 * N, M)

# Suppose C has shape (N,3,M)
N, _, M = total_Coupling.shape
total_Coupling_flat = total_Coupling.reshape(3 * N, M)

print(f'Shapes of the different matrices:\n   U_coupling_shield_inside: {U_coupling_shield_inside.shape}\n   U_coupling_coils_inside: {U_coupling_coils_inside.shape}\n   Coil_Shield_coupling: {Coil_Shield_coupling.shape}\n   B_coupling_shield: {Coupling_shield.shape}\n   secondary_Coupling_shield: {secondary_Coupling_shield.shape}\n   Coupling_coil: {Coupling_coil.shape}\n   total_Coupling: {total_Coupling.shape}\n   total_Coupling_flat: {total_Coupling_flat.shape}')

## Convert coil current into stream function

- In a first step, we find the stream function of the coils making use of
\begin{equation}
    B_{coil} = C_{coil} \cdot \psi_{coil}
\end{equation}
<span style="color:red">Are there improvements to this? I don`t understand the whole code! Why is the Laplace grid used? What do changes in ```lambda_reg``` do?</span>

- ...

- The total magnetic field $B_{tot}$ can be calculated by superpositioning and inserting:
\begin{equation}
    B_{tot} = B_{coil} + B_{shield} = C_{coil} \cdot \psi_{coil} + C_{shield} \cdot \psi_{shield}
\end{equation}


<span style="color:red">Weiß noch nicht wie ich auf ```stream_func_shield``` komme! -> In 2.Paper durchlesen!!!</span>

In [ ]:
def laplacian_2d_grid(n):
    """Build 2D Laplacian for an n x n regular grid, k = i*n + j"""
    M = n * n
    L = np.zeros((M, M))
    
    for i in range(n):
        for j in range(n):
            k = i * n + j
            neighbors = []
            if i > 0:     neighbors.append((i-1) * n + j)   # top
            if i < n-1:   neighbors.append((i+1) * n + j)   # bottom
            if j > 0:     neighbors.append(i * n + (j-1))   # left
            if j < n-1:   neighbors.append(i * n + (j+1))   # right
            
            L[k, k] = -len(neighbors)
            for nb in neighbors:
                L[k, nb] = 1
    return L

n = all_circles.shape[1]
L2 = laplacian_2d_grid(n)          # (n^2, n^2) — one face
L_block = np.kron(np.eye(2), L2)   # (6*n^2, 6*n^2) — all 6 faces # Changed np.eye(6) -> np.eye(2)

# Use in regularized least squares
lambda_reg = 0
A_reg = np.vstack([Coupling_coil_flat, np.sqrt(lambda_reg) * L_block])
b_reg = np.concatenate([B_coil_predicted.reshape(3 * N), np.zeros(2 * n**2)]) # Changed: np.zeros(6 * n**2) -> np.zeros(2 * n**2)

stream_func_coil, _, _, _ = np.linalg.lstsq(A_reg, b_reg, rcond=None) # -> Are there improvements to s? And if yes, how? Chiara said something

stream_func_shield = 

# B-field produced by the shield
B_shield_predicted = np.tensordot(stream_func_shield, Coupling_shield, axes=([0],[2]))
print(B_shield_predicted.shape)

B_tot = B_coil_predicted + B_shield_predicted + B_target_point_no_current

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(
    target_point_coord[:, 0],
    target_point_coord[:, 1],
    target_point_coord[:, 2],
    c=np.linalg.norm(B_tot, axis=1),
    s=100,
    cmap='viridis'
)
fig.colorbar(sc, ax=ax, label=r'$|\mathbf{B}_{\text{total}}|$')